# Lab 3 — Matrix Factorization Recommender with Surprise (SVD)
**ACM Workshop · Recommender Systems · Day 1 · 14:15–15:45 (90 min)**

Just taught: $R\approx PQ^\top$, i.e. $\hat r_{ui}=\mu+b_u+b_i+p_u^\top q_i$, learned by minimising regularised squared error.

**You will build** the main technical deliverable of Day 1:
1. load the *same* train/test split into **Surprise** · 2. train **SVD** · 3. predict & read the predictions
4. evaluate **RMSE / MAE** against baselines and the Lab-2 models · 5. **open the model**: $P$, $Q$, biases
6. **Top-N** recommendations for user 10 · 7. a **hyper-parameter experiment** (`n_factors`, multi-seed) + a small tuning grid
8. **rating prediction ≠ ranking**: Precision@10 / Recall@10 vs a popularity baseline
9. *(optional)* **ALS** from scratch in 25 lines

> Pipeline: Raw → EDA → similarity → CF → **matrix factorization → prediction → top-N → evaluation**

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from surprise import Dataset, Reader, SVD, BaselineOnly, KNNWithMeans, NormalPredictor, accuracy
from surprise.model_selection import GridSearchCV

pd.set_option("display.max_columns", 20, "display.width", 140, "display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def find_proc():
    for base in ["data", "../data", "../../data", "../../../data"]:
        if (Path(base) / "processed" / "train.csv").exists(): return Path(base) / "processed"
    raise FileNotFoundError("Run Lab 1 first (it creates data/processed/train.csv).")
PROC = find_proc(); FIG = PROC.parent.parent / "figures"; FIG.mkdir(exist_ok=True)
train  = pd.read_csv(PROC / "train.csv"); test = pd.read_csv(PROC / "test.csv")
movies = pd.read_csv(PROC / "movies_clean.csv").set_index("movieId"); title = movies.title
import surprise; print("scikit-surprise", surprise.__version__, "| train", len(train), "| test", len(test))

## 1. Convert Pandas → Surprise
Surprise needs a `Reader` that knows the **rating scale**. MovieLens-small uses **0.5–5.0** (the 1–5 scale of MovieLens-100k would be wrong here!).

We reuse the Lab-1 split so that numbers are comparable with Lab 2: build the training set from `train.csv`, and the test set as a list of `(user, item, true_rating)` tuples.

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(train[["userId", "movieId", "rating"]], reader)
trainset = data.build_full_trainset()                       # Surprise's internal structure (maps raw ids -> inner ids)
testset  = list(test[["userId", "movieId", "rating"]].itertuples(index=False, name=None))

print("users:", trainset.n_users, "| items:", trainset.n_items, "| ratings:", trainset.n_ratings)
print("global mean μ:", round(trainset.global_mean, 4))
print("first test tuple (uid, iid, true rating):", testset[0])

## 2. Baselines — what must we beat?
- `NormalPredictor`: random ratings drawn from the training distribution (floor).
- `BaselineOnly`: $\hat r_{ui}=\mu+b_u+b_i$ — **no personalisation beyond biases**, fitted by ALS. Surprisingly strong!

In [ ]:
results = {}
def run(name, algo):
    t = time.time(); algo.fit(trainset); pred = algo.test(testset)
    results[name] = {"RMSE": accuracy.rmse(pred, verbose=False), "MAE": accuracy.mae(pred, verbose=False), "fit+predict (s)": round(time.time() - t, 1)}
    return algo, pred

run("random (NormalPredictor)", NormalPredictor())
run("baseline  μ + b_u + b_i", BaselineOnly(verbose=False))
pd.DataFrame(results).T.round(4)

## 3. Train SVD
```python
SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, biased=True, random_state=...)
```
| parameter | meaning |
|---|---|
| `n_factors` | dimension $k$ of $p_u,q_i$ — model capacity |
| `n_epochs` | passes of SGD over the training ratings |
| `lr_all` | learning rate (step size) |
| `reg_all` | $\lambda$, L2 regularisation strength |
| `biased` | include $\mu, b_u, b_i$ (True by default) |

In [ ]:
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd, predictions = run("SVD (50 factors)", svd)      # fit on trainset, then predict testset
print("predictions is a list of", len(predictions), "Prediction objects. The first one:\n", predictions[0])

## 4. Look at the predictions

In [ ]:
pred_df = pd.DataFrame([(p.uid, p.iid, p.r_ui, p.est) for p in predictions], columns=["userId", "movieId", "actual", "predicted"])
pred_df["title"] = title.reindex(pred_df.movieId).values
pred_df["abs_error"] = (pred_df.actual - pred_df.predicted).abs()
print("A random sample of test predictions:")
display(pred_df.sample(8, random_state=1).round(2))
print("\nBiggest mistakes:")
pred_df.nlargest(5, "abs_error").round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(pred_df.actual + np.random.default_rng(0).uniform(-.12, .12, len(pred_df)), pred_df.predicted, s=2, alpha=.15, color="#2b6cb0")
ax[0].plot([0.5, 5], [0.5, 5], "r--"); ax[0].set(xlabel="actual rating", ylabel="predicted rating", title="Predicted vs actual (jittered)")
ax[1].hist(pred_df.actual - pred_df.predicted, bins=50, color="#38a169"); ax[1].set(title="Error distribution (actual − predicted)", xlabel="error")
plt.tight_layout(); plt.savefig(FIG / "lab3_pred_vs_actual.png"); plt.show()

**Read the plot:** predictions are *compressed towards the middle* (the model rarely says 0.5 or 5.0 — it is hedging, which minimises squared error). That is why RMSE ≈ 0.88 although ratings span 0.5–5.

## 5. Evaluate: RMSE and MAE
$$\mathrm{RMSE}=\sqrt{\tfrac1N\sum(r-\hat r)^2}\qquad \mathrm{MAE}=\tfrac1N\sum|r-\hat r|$$
RMSE punishes large errors more; MAE is the typical error in "stars".

In [ ]:
print("SVD  RMSE:", round(accuracy.rmse(predictions, verbose=False), 4), "| MAE:", round(accuracy.mae(predictions, verbose=False), 4))
# Compare with the Lab-2 style neighbourhood models, now via Surprise's built-in KNN
run("user-KNN (k=40, Pearson)", KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": True}, verbose=False))
run("item-KNN (k=40, cosine)",  KNNWithMeans(k=40, sim_options={"name": "cosine",  "user_based": False}, verbose=False))
tab = pd.DataFrame(results).T; tab.round(4)

**Honest reading of this table** — this is a *teaching point*, not a flaw:
- All personalised models land close to each other; the **bias baseline** already captures most of the signal.
- SVD gains over the baseline are *small* (hundredths of RMSE) on this small dataset. Netflix-Prize-scale gains came from huge data, tuning and ensembles.
- Therefore: **always report a baseline**, and don't trust a 0.002 difference without repeating over seeds (see §8).

## 6. Open the model: what did SVD learn?
$P$ = `svd.pu` (users × k), $Q$ = `svd.qi` (items × k), biases `svd.bu`, `svd.bi`.

In [ ]:
print("P (users×factors):", svd.pu.shape, "| Q (items×factors):", svd.qi.shape, "| μ =", round(svd.trainset.global_mean, 3))
print("Parameters learned:", svd.pu.size + svd.qi.size + len(svd.bu) + len(svd.bi), "vs the full matrix:", trainset.n_users * trainset.n_items)

# Re-compute one prediction by hand:  μ + b_u + b_i + p_u · q_i
uid, iid = 10, 1                                   # user 10, Toy Story
iu, ii = trainset.to_inner_uid(uid), trainset.to_inner_iid(iid)
manual = svd.trainset.global_mean + svd.bu[iu] + svd.bi[ii] + svd.pu[iu] @ svd.qi[ii]
print(f"\nuser {uid} × '{title[iid]}':  by hand = {manual:.4f}   svd.predict = {svd.predict(uid, iid).est:.4f}")
print(f"   μ={svd.trainset.global_mean:.2f}  b_u={svd.bu[iu]:+.2f}  b_i={svd.bi[ii]:+.2f}  p_u·q_i={svd.pu[iu] @ svd.qi[ii]:+.2f}")

In [ ]:
# Biases: the most "generous" movies and harshest users
raw_i = np.array([trainset.to_raw_iid(k) for k in range(trainset.n_items)]); cnt = np.bincount([i for (_, i, _) in trainset.all_ratings()], minlength=trainset.n_items)
bias_df = pd.DataFrame({"movieId": raw_i, "b_i": svd.bi, "n": cnt}); bias_df["title"] = title.reindex(bias_df.movieId).values
print("Movies with the highest item bias (≥ 50 ratings):"); display(bias_df[bias_df.n >= 50].nlargest(5, "b_i")[["title", "b_i", "n"]].round(2))
print("…and the lowest:"); bias_df[bias_df.n >= 50].nsmallest(5, "b_i")[["title", "b_i", "n"]].round(2)

### Latent factors → "similar movies" without any genre information
Items are now **vectors**; similar movies = nearby vectors (cosine on rows of $Q$). Compare with Lab 2's neighbours.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
Q = svd.qi; counts = pd.Series(cnt, index=raw_i)
def similar_by_factors(movie_id, n=8, min_n=30):
    s = cosine_similarity(Q[[trainset.to_inner_iid(movie_id)]], Q)[0]
    out = pd.DataFrame({"movieId": raw_i, "cos": s, "n": cnt}); out = out[(out.n >= min_n) & (out.movieId != movie_id)]
    out["title"] = title.reindex(out.movieId).values
    return out.nlargest(n, "cos")[["title", "cos", "n"]].round(3)
for mid in (1, 2571):
    print("Closest to", title[mid]); display(similar_by_factors(mid))

In [ ]:
# 2-D picture of the item space (PCA of Q for the 300 most-rated movies), coloured by main genre
from sklearn.decomposition import PCA
top = np.argsort(-cnt)[:300]; xy = PCA(2, random_state=0).fit_transform(Q[top])
main = movies.reindex(raw_i[top]).genres.str.split("|").str[0].values
fig, ax = plt.subplots(figsize=(7, 5)); genres = ["Drama", "Comedy", "Action", "Adventure", "Thriller", "Crime", "Sci-Fi", "Animation"]
for gname in genres:
    m_ = main == gname; ax.scatter(xy[m_, 0], xy[m_, 1], s=22, label=gname, alpha=.8)
ax.scatter(xy[~np.isin(main, genres), 0], xy[~np.isin(main, genres), 1], s=10, c="lightgray", label="other")
for mid in (1, 2571, 318, 296, 4993, 1721):
    j = np.where(raw_i[top] == mid)[0]
    if len(j): ax.annotate(title[mid].split(" (")[0], xy[j[0]], fontsize=8)
ax.legend(fontsize=7, ncol=2); ax.set(title="Item factors Q projected to 2-D (300 most-rated movies)", xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.savefig(FIG / "lab3_item_space.png"); plt.show()

**Compare with Lab 2:** adjusted-cosine neighbours were cleaner than these. The latent space is a *compression* trained to predict ratings, not to be a similarity engine — expect some surprises (and some real structure).

⚠️ **Caution (say this out loud):** latent dimensions are *not* guaranteed to equal genres. Any rotation of the factors gives the same predictions, so individual axes have no fixed meaning — only the *geometry* (who is near whom) is meaningful.

## 7. Top-N recommendations
Prediction → **ranking**: predict a rating for every movie the user has *not* rated, sort, take the top N.
```
User → candidate movies (unrated) → predict rating for each → sort → Top-N
```

In [ ]:
all_iids = list(trainset._raw2inner_id_items.keys())     # every movie in the training data
counts_by_raw = counts

def top_n(algo, uid, n=5, min_pop=20):
    seen = {trainset.to_raw_iid(i) for i, _ in trainset.ur[trainset.to_inner_uid(uid)]}
    cands = [i for i in all_iids if i not in seen and counts_by_raw[i] >= min_pop]   # the same support guard as Lab 2
    scored = sorted(((i, algo.predict(uid, i).est) for i in cands), key=lambda t: -t[1])[:n]
    return pd.DataFrame({"title": [title[i] for i, _ in scored], "predicted": [round(p, 2) for _, p in scored],
                         "# ratings": [counts_by_raw[i] for i, _ in scored]}, index=[i for i, _ in scored])

print("Recommended for user 10 (SVD, 50 factors):"); top_n(svd, 10)

In [ ]:
print("User 10's own favourites in training, for comparison:")
u10 = train[train.userId == 10].nlargest(8, "rating"); u10.assign(title=title.reindex(u10.movieId).values)[["title", "rating"]]

In [ ]:
# Three different users → three different lists (personalisation!)
for u in (10, 414, 77):
    print(f"\n=== user {u} ({(train.userId == u).sum()} training ratings) ==="); display(top_n(svd, u, n=5))

## 8. Experiment: how many latent factors?  (research-style)
> *Does increasing `n_factors` always improve performance?*

One run per setting is **noise** (random initialisation) → repeat over 5 seeds and report **mean ± std**.

In [ ]:
rows = []
for k in (5, 20, 50, 100, 200):
    for seed in range(5):
        m = SVD(n_factors=k, random_state=seed).fit(trainset); p = m.test(testset)
        rows.append({"n_factors": k, "seed": seed, "RMSE": accuracy.rmse(p, verbose=False), "MAE": accuracy.mae(p, verbose=False)})
exp = pd.DataFrame(rows); summ = exp.groupby("n_factors").agg(RMSE=("RMSE", "mean"), RMSE_std=("RMSE", "std"), MAE=("MAE", "mean"))
display(summ.round(4))
fig, ax = plt.subplots(figsize=(5.8, 3.4)); ax.errorbar(summ.index, summ.RMSE, yerr=summ.RMSE_std, fmt="o-", capsize=3, color="#2b6cb0")
ax.axhline(results["baseline  μ + b_u + b_i"]["RMSE"], ls="--", color="gray", label="bias baseline"); ax.legend()
ax.set(xlabel="n_factors", ylabel="test RMSE", title="Effect of latent dimension (mean ± std over 5 seeds)", xscale="log")
plt.tight_layout(); plt.savefig(FIG / "lab3_nfactors.png"); plt.show()

**Discuss:** the curve is *flat or slowly rising* — more factors ≠ better. More capacity means more room to **overfit** sparse ratings (about 130 ratings per user, 8 per movie). The remedy is *regularisation* (`reg_all`), not just fewer factors. Let's tune it.

In [ ]:
# Small grid search with 3-fold CV on the TRAINING set only (the test set stays untouched!)
grid = {"n_factors": [20, 50], "n_epochs": [20, 40], "lr_all": [0.005, 0.01], "reg_all": [0.02, 0.05, 0.1]}
gs = GridSearchCV(SVD, grid, measures=["rmse"], cv=3, n_jobs=-1, joblib_verbose=0); gs.fit(data)
print("best CV RMSE:", round(gs.best_score["rmse"], 4), "with", gs.best_params["rmse"])
cv = pd.DataFrame(gs.cv_results)[["param_n_factors", "param_n_epochs", "param_lr_all", "param_reg_all", "mean_test_rmse"]].sort_values("mean_test_rmse")
cv.head(6).round(4)

In [ ]:
best = SVD(**gs.best_params["rmse"], random_state=42)
best, _ = run("SVD tuned (CV on train)", best)
pd.DataFrame(results).T.round(4)

Tuning `reg_all` helps more than adding factors. **Never tune on the test set**: we picked the parameters with cross-validation on *train*, then looked at test once.

## 9. Rating prediction ≠ Top-N recommendation
A model with a great RMSE can still produce a poor top-10 list. Evaluate the *ranking* directly:
for every test user, rank all their unseen movies, take the top 10, and ask how many are movies the user later **liked (rating ≥ 4)** in the test set.

- **Precision@10** = (# liked items in top-10) / 10 · **Recall@10** = (# liked in top-10) / (# liked in test)
- We compare SVD with a **popularity** recommender (same list for everyone). *Candidate set = all movies with ≥ 20 training ratings.*

In [ ]:
THRESH, K = 4.0, 10
liked = test[test.rating >= THRESH].groupby("userId").movieId.apply(set)
train_seen = train.groupby("userId").movieId.apply(set)
cands = [i for i in all_iids if counts_by_raw[i] >= 20]
pop = train.groupby("movieId").rating.agg(["mean", "count"]); pop = pop.loc[cands]
pop["score"] = (pop["count"] * pop["mean"] + 30 * train.rating.mean()) / (pop["count"] + 30)
pop_rank = list(pop.sort_values("score", ascending=False).index)

def pr_at_k(rank_fn, users, K=K):
    P, R_ = [], []
    for u in users:
        rec = rank_fn(u)[:K]; hit = len(set(rec) & liked[u])
        P.append(hit / K); R_.append(hit / len(liked[u]))
    return np.mean(P), np.mean(R_)

eval_users = [u for u in liked.index if len(liked[u]) >= 3][:300]          # 300 users keeps it quick
def rank_pop(u): return [i for i in pop_rank if i not in train_seen[u]]
def rank_model(algo):
    def f(u):
        s = [(i, algo.predict(u, i).est) for i in cands if i not in train_seen[u]]
        return [i for i, _ in sorted(s, key=lambda t: -t[1])]
    return f

t = time.time()
rank_res = pd.DataFrame({"popularity (non-personalised)": pr_at_k(rank_pop, eval_users),
                         "SVD (50 factors)": pr_at_k(rank_model(svd), eval_users),
                         "SVD tuned": pr_at_k(rank_model(best), eval_users)}, index=[f"Precision@{K}", f"Recall@{K}"]).T
print(f"evaluated {len(eval_users)} users in {time.time() - t:.0f}s"); rank_res.round(4)

**Why does SVD lose here?** (1) It is trained to minimise *rating* error, not to *rank*; (2) candidates with few ratings can get inflated predictions (noisy item factors); (3) "not in the test set" ≠ "disliked" — unobserved items are *missing not at random*, which favours popular items. Try raising the candidate threshold to ≥ 100 ratings and watch Precision@10 change.

**Interpretation.** Popularity is a *brutally* strong top-N baseline on MovieLens (popular movies are watched *and* liked by most). A personalised model that wins on RMSE does not automatically win on Precision@10 — **they measure different things**:

| | rating prediction | Top-N recommendation |
|---|---|---|
| question | "what stars will she give?" | "which 10 should we show?" |
| metric | RMSE / MAE | Precision@N, Recall@N, NDCG |
| depends on | accuracy of *every* prediction | *ordering* at the top |

Real systems also care about diversity, novelty and fairness — hints for Day 2.

## 10. (Optional) ALS from scratch
**ALS**: fix $Q$ → each user vector $p_u$ is a *ridge regression* with a closed form; fix $P$ → same for each item; alternate.
$$p_u=\Big(\sum_{i\in\Omega_u}q_iq_i^\top+\lambda I\Big)^{-1}\sum_{i\in\Omega_u}r_{ui}q_i$$
No learning rate needed; each half-step solves a small $k\times k$ system exactly.

In [ ]:
def als(train_df, k=20, lam=10.0, iters=12, seed=0):
    uids = np.sort(train_df.userId.unique()); iids = np.sort(train_df.movieId.unique())
    ui = {u: n for n, u in enumerate(uids)}; ii = {i: n for n, i in enumerate(iids)}
    u_ = train_df.userId.map(ui).values; i_ = train_df.movieId.map(ii).values; r_ = train_df.rating.values
    mu = r_.mean(); r0 = r_ - mu                                   # centre globally; ALS learns the rest
    by_u = [np.where(u_ == n)[0] for n in range(len(uids))]; by_i = [np.where(i_ == n)[0] for n in range(len(iids))]
    rng = np.random.default_rng(seed); P = rng.normal(0, .1, (len(uids), k)); Q = rng.normal(0, .1, (len(iids), k)); I = np.eye(k)
    hist = []
    for it in range(iters):
        for n, idx in enumerate(by_u):                              # fix Q, solve each user
            Qi = Q[i_[idx]]; P[n] = np.linalg.solve(Qi.T @ Qi + lam * I, Qi.T @ r0[idx])
        for n, idx in enumerate(by_i):                              # fix P, solve each item
            Pu = P[u_[idx]]; Q[n] = np.linalg.solve(Pu.T @ Pu + lam * I, Pu.T @ r0[idx])
        hist.append(np.sqrt(np.mean((r0 - (P[u_] * Q[i_]).sum(1)) ** 2)))
    return dict(P=P, Q=Q, mu=mu, ui=ui, ii=ii, hist=hist)

def als_predict(m, df):
    out = np.full(len(df), m["mu"])
    ok = df.userId.isin(m["ui"]).values & df.movieId.isin(m["ii"]).values
    uu = df.userId[ok].map(m["ui"]).values; vv = df.movieId[ok].map(m["ii"]).values
    out[ok] = m["mu"] + (m["P"][uu] * m["Q"][vv]).sum(1)
    return out.clip(0.5, 5)

t = time.time(); m_als = als(train, k=20, lam=10.0, iters=12)
p = als_predict(m_als, test)
print(f"ALS (k=20, λ=10): test RMSE = {np.sqrt(np.mean((test.rating - p) ** 2)):.4f}  MAE = {np.mean(np.abs(test.rating - p)):.4f}   [{time.time() - t:.0f}s]")
print("train RMSE per sweep:", np.round(m_als['hist'], 3))

Training error falls monotonically with every sweep (each half-step is an exact minimiser — **the objective can never go up**). With no bias terms this plain ALS is less accurate than biased SVD: add $b_u,b_i$ (or use `BaselineOnly` first) as an exercise. Try `lam` ∈ {1, 10, 30}: too small overfits, too large underfits.

## 11. 🏁 Final challenge — *independently*
> **For a given user, build a recommender that recommends five movies the user has not rated.**

Write `recommend(user_id, n=5)` using any model from today, return titles, and justify your choice of model in one sentence.

In [ ]:
# TODO: write your code here


## 12. Summary table & Day-1 pipeline

In [ ]:
final = pd.DataFrame(results).T[["RMSE", "MAE"]].round(4); final.sort_values("RMSE")

```
Raw ratings ─▶ EDA ─▶ user–item matrix ─▶ similarity ─▶ neighbourhood CF ─▶ matrix factorization (SVD / ALS)
                                                                         ─▶ rating prediction ─▶ Top-N ─▶ RMSE/MAE, Precision@N
```
## ✅ Lab 3 deliverable — checklist
- [x] Surprise dataset & split · [x] SVD trained · [x] predictions table · [x] RMSE & MAE vs baselines
- [x] inspected $P$, $Q$, biases · [x] Top-5 for user 10 · [x] `n_factors` experiment (5 seeds) · [x] tuning with CV
- [x] Precision@10 vs popularity · [x] (optional) ALS

**Next:** Day 2 builds on this pipeline.